In [1]:
# 1. Preparar el entorno
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 2. Cargar el dataset
dataset = pd.read_csv('/workspace/diabetes.csv')

# Revisa el contenido del dataset
print(dataset.info())
print(dataset.describe())

# 3. Preparar el dataset para la clasificación
# Seleccionar las características (features) y el target (Outcome)
X = dataset.drop(columns=['Outcome'])
y = dataset['Outcome']

# Dividir el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Normalizar las características
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convertir los datos en tensores para PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# 4. Construcción de la red neuronal
class FeedForwardNN(nn.Module):
    def __init__(self):
        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 64)  # Capa de entrada (número de características)
        self.fc2 = nn.Linear(64, 32)  # Capa oculta 1
        self.fc3 = nn.Linear(32, 1)  # Capa de salida (una neurona para clasificación binaria)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))  # Activación ReLU en la primera capa
        x = torch.relu(self.fc2(x))  # Activación ReLU en la segunda capa
        x = torch.sigmoid(self.fc3(x))  # Activación sigmoide para la salida
        return x

# Crear una instancia del modelo
model = FeedForwardNN()

# 5. Entrenamiento del modelo
# Definir la función de pérdida y el optimizador
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss para clasificación binaria
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Optimizador Adam

# Entrenar el modelo
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()  # Reiniciar los gradientes
    outputs = model(X_train_tensor)  # Hacer predicciones con los datos de entrenamiento
    loss = criterion(outputs, y_train_tensor)  # Calcular la pérdida
    loss.backward()  # Hacer la retropropagación
    optimizer.step()  # Actualizar los parámetros del modelo

    # Mostrar la pérdida cada 10 épocas
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 6. Evaluación del modelo
# El modelo está en modo evaluación
model.eval()

# Hacer predicciones en el conjunto de prueba
with torch.no_grad():
    y_pred_probs = model(X_test_tensor).detach().numpy()  # Predicciones de probabilidades
    y_pred = (y_pred_probs >= 0.5).astype(int)  # Convertir probabilidades en 0 o 1

# Calcular las métricas de evaluación
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

# Mostrar las métricas
print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')
print(report)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std    